### Function calling: el modelo pide, vos ejecutás

Un LLM solo genera texto — no puede consultar una base de datos real ni ejecutar acciones. Function calling le da al modelo una lista de herramientas disponibles (funciones con nombre, descripción y parámetros). El modelo NUNCA ejecuta código directo: decide cuándo necesita una herramienta y con qué argumentos, vos ejecutás la función de verdad en tu código, y le devolvés el resultado para que arme la respuesta final en lenguaje natural.

Flujo completo: prompt + tools → modelo pide llamar función X → ejecutás X → le devolvés el resultado → modelo arma respuesta final usando ese dato real.

In [1]:
import ollama

def precio_vuelo(ciudad_destino: str) -> str:
    precios = {"madrid": 850, "roma": 720, "tokio": 1400}
    precio = precios.get(ciudad_destino.lower(), "desconocido")
    return f"El vuelo a {ciudad_destino} cuesta {precio} USD"

tools = [{
    "type": "function",
    "function": {
        "name": "precio_vuelo",
        "description": "Devuelve el precio de un vuelo a una ciudad destino",
        "parameters": {
            "type": "object",
            "properties": {
                "ciudad_destino": {"type": "string", "description": "Ciudad a la que se quiere viajar"},
            },
            "required": ["ciudad_destino"],
        },
    },
}]

messages = [{"role": "user", "content": "¿Cuánto cuesta un vuelo a Roma?"}]
response = ollama.chat(model="llama3.2", messages=messages, tools=tools)

print("¿Pidió llamar una función?", response["message"].get("tool_calls"))


¿Pidió llamar una función? [ToolCall(function=Function(name='precio_vuelo', arguments={'ciudad_destino': 'Roma'}))]


El modelo pidió la función bien — con el argumento correcto. Ahora completamos el ciclo: ejecutamos `precio_vuelo` de verdad, le devolvemos el resultado como mensaje `role="tool"`, y le pedimos que arme la respuesta final.

In [2]:
messages.append(response["message"])

for tool_call in response["message"].get("tool_calls", []):
    resultado = precio_vuelo(**tool_call["function"]["arguments"])
    messages.append({"role": "tool", "content": resultado, "tool_name": tool_call["function"]["name"]})

respuesta_final = ollama.chat(model="llama3.2", messages=messages)
print(respuesta_final["message"]["content"])


Lo siento, pero no puedo proporcionar un precio exacto para un vuelo a Roma, ya que el precio puede variar dependiendo de varios factores como la temporada, la aerolínea, la fecha de viaje, el tipo de asiento y la disponibilidad.

Sin embargo, puedo darte algunas ideas generales sobre cómo puedes encontrar un vuelo a Roma a un precio razonable:

1. **Investiga y compara precios**: Utiliza herramientas de búsqueda de vuelos en línea como Google Flights, Skyscanner, Kayak o Expedia para comparar precios y encontrar las mejores ofertas.
2. **Busca vuelos económicos**: Los vuelos económicos suelen ser más baratos que los vuelos de clase alta. Considera optar por un vuelo económico con una aerolínea económica.
3. **Considera la temporada**: Los precios de los vuelos pueden variar dependiendo de la temporada. Los vuelos durante la temporada baja (generalmente enero a marzo y noviembre a diciembre) pueden ser más baratos que los vuelos durante la temporada alta (junio a agosto y diciembre a e

**Qué muestra esto:** el primer paso (decidir qué función llamar y con qué argumento) salió perfecto. El segundo paso — usar el resultado real que le devolvimos — falló: llama3.2 (3B) ignoró el dato ("El vuelo a Roma cuesta 720 USD") y alucinó un rango de precios inventado. Mismo patrón que vimos con la ventana de contexto: el mecanismo está bien implementado, pero un modelo chico no garantiza seguir el protocolo de forma confiable. Comparamos contra un modelo frontier a continuación.

In [4]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents="¿Cuánto cuesta un vuelo a Roma?",
    config=types.GenerateContentConfig(tools=[precio_vuelo]),
)
print(response.text)


El precio de un vuelo a Roma es de 720 USD.


**Diferencia clave:** con Gemini alcanza `tools=[precio_vuelo]` — le pasás la función Python directo, el SDK detecta la firma solo y hace todo el ciclo automático (AFC — automatic function calling): decide llamarla, la ejecuta, y arma la respuesta final con el dato real (720 USD, correcto). Menos código, y el modelo grande sí respeta el resultado de la herramienta en vez de alucinar. Para el Proyecto 2 (agente de aerolínea) esto importa: si el modelo ignora los datos reales de disponibilidad/precio, el agente miente.